# 🤖 MiniGPT — Trained on STEMP21 Website Data
Run each cell **top to bottom**. At the end you will download `minigpt.pt`.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install tiktoken -q

In [ ]:
# ── Cell 2: Upload your stemp21_data.txt file ─────────────────────────────
from google.colab import files

print('A file picker will appear below. Click it and upload stemp21_data.txt')
uploaded = files.upload()

with open('stemp21_data.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f'\n✓ Loaded {len(text):,} characters of text')
print('Preview:', text[:300])

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
# ── Cell 4: Hyperparameters ───────────────────────────────────────────────
# Small model because stemp21 website has limited text
VOCAB_SIZE  = 50257
BLOCK_SIZE  = 128
N_EMBED     = 128
N_HEAD      = 4
N_LAYER     = 4
DROPOUT     = 0.1

BATCH_SIZE  = 16
MAX_ITERS   = 3000
EVAL_EVERY  = 300
LEARNING_RATE = 3e-4

print('Hyperparameters set.')

In [ ]:
# ── Cell 5: Tokenize the data ─────────────────────────────────────────────
enc = tiktoken.get_encoding('gpt2')

tokens = enc.encode(text)
data   = torch.tensor(tokens, dtype=torch.long)

# 90% train, 10% validation
n      = int(0.9 * len(data))
train  = data[:n]
val    = data[n:]

print(f'Total tokens : {len(data):,}')
print(f'Train tokens : {len(train):,}')
print(f'Val tokens   : {len(val):,}')

In [ ]:
# ── Cell 6: Data loader ───────────────────────────────────────────────────
def get_batch(split):
    d  = train if split == 'train' else val
    ix = torch.randint(len(d) - BLOCK_SIZE, (BATCH_SIZE,))
    x  = torch.stack([d[i:i+BLOCK_SIZE]   for i in ix])
    y  = torch.stack([d[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)

print('Data loader ready.')

In [ ]:
# ── Cell 7: Model Architecture ────────────────────────────────────────────
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(N_EMBED, head_size, bias=False)
        self.query = nn.Linear(N_EMBED, head_size, bias=False)
        self.value = nn.Linear(N_EMBED, head_size, bias=False)
        self.dropout = nn.Dropout(DROPOUT)
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        head_size = k.shape[-1]
        wei = q @ k.transpose(-2, -1) * (head_size ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v   = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        head_size    = N_EMBED // N_HEAD
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(N_HEAD)])
        self.proj    = nn.Linear(N_EMBED, N_EMBED)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_EMBED, 4 * N_EMBED),
            nn.GELU(),
            nn.Linear(4 * N_EMBED, N_EMBED),
            nn.Dropout(DROPOUT)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa  = MultiHeadAttention()
        self.ff  = FeedForward()
        self.ln1 = nn.LayerNorm(N_EMBED)
        self.ln2 = nn.LayerNorm(N_EMBED)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(VOCAB_SIZE, N_EMBED)
        self.pos_emb   = nn.Embedding(BLOCK_SIZE, N_EMBED)
        self.blocks    = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.ln_f      = nn.LayerNorm(N_EMBED)
        self.lm_head   = nn.Linear(N_EMBED, VOCAB_SIZE, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

model = MiniGPT().to(device)
params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'✓ Model created: {params:.1f}M parameters')

In [ ]:
# ── Cell 8: Training loop ─────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        L = []
        for _ in range(20):
            x, y = get_batch(split)
            _, loss = model(x, y)
            L.append(loss.item())
        losses[split] = np.mean(L)
    model.train()
    return losses

print('Starting training...')
for step in range(MAX_ITERS):
    if step % EVAL_EVERY == 0:
        losses = estimate_loss()
        print(f'Step {step:4d} | train loss: {losses["train"]:.4f} | val loss: {losses["val"]:.4f}')

    x, y = get_batch('train')
    logits, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print('\n✓ Training complete!')

In [ ]:
# ── Cell 9: Test generation ───────────────────────────────────────────────
model.eval()

def generate(prompt, max_tokens=100, temperature=0.8, top_k=40):
    input_ids = torch.tensor(enc.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        for _ in range(max_tokens):
            idx_cond = input_ids[:, -BLOCK_SIZE:]
            logits, _ = model(idx_cond)
            logits = logits[:, -1, :] / temperature
            v, ix  = torch.topk(logits, min(top_k, logits.size(-1)))
            probs  = F.softmax(v, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            next_tok = torch.gather(ix, -1, next_idx)
            input_ids = torch.cat([input_ids, next_tok], dim=1)
    return enc.decode(input_ids[0].tolist())

output = generate('STEMP21 is', max_tokens=100)
print(output)

In [ ]:
# ── Cell 10: Save and download model ──────────────────────────────────────
torch.save(model.state_dict(), 'minigpt.pt')
print('✓ Saved minigpt.pt')

files.download('minigpt.pt')
print('✓ Download started!')